In [1]:
import pandas as pd

file_path = 'yellow_tripdata_2021-01.csv.gz'

df_sample = pd.read_csv(file_path, nrows=100, compression='gzip')

print("=== First 5 rows ===")
print(df_sample.head())

print("\n=== Data types ===")
print(df_sample.dtypes)

print("\n=== Shape of sample ===")
print(df_sample.shape)

=== First 5 rows ===
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amo

In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_path = 'yellow_tripdata_2021-01.csv.gz'
chunk_size = 100000

total_rows = 0
null_counts = {}
min_vals = {}
max_vals = {}
dtypes_dict = {}

for chunk in pd.read_csv(file_path, chunksize=chunk_size, compression='gzip', low_memory=False):
    total_rows += len(chunk)
    
    for col in chunk.columns:
        if col not in dtypes_dict:
            dtypes_dict[col] = chunk[col].dtype
            null_counts[col] = 0
            if pd.api.types.is_numeric_dtype(chunk[col]):
                min_vals[col] = float('inf')
                max_vals[col] = float('-inf')
        
        null_counts[col] += chunk[col].isnull().sum()
        
        if pd.api.types.is_numeric_dtype(chunk[col]):
            if col not in min_vals:
                min_vals[col] = chunk[col].min()
                max_vals[col] = chunk[col].max()
            else:
                min_vals[col] = min(min_vals[col], chunk[col].min())
                max_vals[col] = max(max_vals[col], chunk[col].max())

print(f"Total rows: {total_rows}\n")

print(f"{'Column':<30} {'Dtype':<12} {'Nulls':<10} {'Null%':<10} {'Min':<15} {'Max':<15}")
print("-" * 95)

for col in dtypes_dict.keys():
    dtype = dtypes_dict[col]
    nulls = null_counts.get(col, 0)
    null_pct = (nulls / total_rows * 100) if total_rows > 0 else 0
    
    if col in min_vals:
        min_val = f"{min_vals[col]:.2f}"
        max_val = f"{max_vals[col]:.2f}"
    else:
        min_val = "N/A"
        max_val = "N/A"
    
    print(f"{col:<30} {str(dtype):<12} {nulls:<10} {null_pct:<10.2f} {min_val:<15} {max_val:<15}")

Total rows: 1369765

Column                         Dtype        Nulls      Null%      Min             Max            
-----------------------------------------------------------------------------------------------
VendorID                       int64        98352      7.18       1.00            2.00           
tpep_pickup_datetime           str          0          0.00       N/A             N/A            
tpep_dropoff_datetime          str          0          0.00       N/A             N/A            
passenger_count                int64        98352      7.18       0.00            8.00           
trip_distance                  float64      0          0.00       0.00            263163.28      
RatecodeID                     int64        98352      7.18       1.00            99.00          
store_and_fwd_flag             str          98352      7.18       nan             nan            
PULocationID                   int64        0          0.00       1.00            265.00         
D

In [5]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_path = 'yellow_tripdata_2021-01.csv.gz'
chunk_size = 100000

problems = {}

for chunk in pd.read_csv(file_path, chunksize=chunk_size, compression='gzip', low_memory=False):
    chunk['tpep_pickup_datetime'] = pd.to_datetime(chunk['tpep_pickup_datetime'], errors='coerce')
    chunk['tpep_dropoff_datetime'] = pd.to_datetime(chunk['tpep_dropoff_datetime'], errors='coerce')
    
    bad_dates = ((chunk['tpep_pickup_datetime'] < '2021-01-01') | 
                 (chunk['tpep_pickup_datetime'] >= '2021-02-01')).sum()
    problems['bad_dates'] = problems.get('bad_dates', 0) + bad_dates
    
    bad_fares = (chunk['fare_amount'] <= 0).sum()
    problems['bad_fares'] = problems.get('bad_fares', 0) + bad_fares
    
    bad_distances = (chunk['trip_distance'] <= 0).sum()
    problems['bad_distances'] = problems.get('bad_distances', 0) + bad_distances
    
    zero_passengers = (chunk['passenger_count'] == 0).sum()
    problems['zero_passengers'] = problems.get('zero_passengers', 0) + zero_passengers
    
    problems['duplicates'] = problems.get('duplicates', 0) + chunk.duplicated().sum()

print("=== Problems Found ===\n")
for problem, count in problems.items():
    print(f"{problem:<25}: {count} rows")

=== Problems Found ===

bad_dates                : 24 rows
bad_fares                : 7411 rows
bad_distances            : 19952 rows
zero_passengers          : 26726 rows
duplicates               : 0 rows


In [6]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_path = 'yellow_tripdata_2021-01.csv.gz'

def clean(df):
    cleaned_df = df.copy()
    rows_removed = {}
    initial_len = len(cleaned_df)
    
    # Rule 1: Remove duplicates
    dup_count = cleaned_df.duplicated().sum()
    cleaned_df = cleaned_df.drop_duplicates()
    rows_removed['duplicates'] = dup_count
    
    # Rule 2: Remove zero passengers
    zero_pass = (cleaned_df['passenger_count'] == 0).sum()
    cleaned_df = cleaned_df[cleaned_df['passenger_count'] > 0]
    rows_removed['zero_passengers'] = zero_pass
    
    # Rule 3: Remove bad fares and distances
    bad_fare_dist = ((cleaned_df['fare_amount'] <= 0) | (cleaned_df['trip_distance'] <= 0)).sum()
    cleaned_df = cleaned_df[(cleaned_df['fare_amount'] > 0) & (cleaned_df['trip_distance'] > 0)]
    rows_removed['bad_fare_or_distance'] = bad_fare_dist
    
    # Rule 4: Convert dates and remove bad dates
    cleaned_df['tpep_pickup_datetime'] = pd.to_datetime(cleaned_df['tpep_pickup_datetime'], errors='coerce')
    bad_dates = ((cleaned_df['tpep_pickup_datetime'] < '2021-01-01') | 
                 (cleaned_df['tpep_pickup_datetime'] >= '2021-02-01')).sum()
    cleaned_df = cleaned_df[(cleaned_df['tpep_pickup_datetime'] >= '2021-01-01') & 
                            (cleaned_df['tpep_pickup_datetime'] < '2021-02-01')]
    rows_removed['out_of_jan_2021'] = bad_dates
    
    rows_removed['total_removed'] = initial_len - len(cleaned_df)
    
    return cleaned_df, rows_removed

# Load full dataset
print("Loading full dataset...")
df_full = pd.read_csv(file_path, compression='gzip', low_memory=False)

print(f"\nRows before cleaning: {len(df_full)}")

# Apply cleaning
cleaned_df, removed_dict = clean(df_full)

print(f"Rows after cleaning: {len(cleaned_df)}")
print(f"\n=== Rows removed per rule ===")
for rule, count in removed_dict.items():
    print(f"{rule:<30}: {count}")

Loading full dataset...

Rows before cleaning: 1369765
Rows after cleaning: 1220104

=== Rows removed per rule ===
duplicates                    : 0
zero_passengers               : 26726
bad_fare_or_distance          : 24560
out_of_jan_2021               : 23
total_removed                 : 149661
